# SFT training (supervised fine-tuning)

**SFT** trains on labeled question–answer pairs using a cross-entropy loss. **GRPO** (see [05_grpo_training.ipynb](05_grpo_training.ipynb)) uses rollouts and rewards for forecasting-style tasks.

This notebook follows the same flow as the GRPO tutorial: load a dataset, split into train/test, estimate cost, and run a hosted training job with `SFTTrainingConfig`.

## Install the SDK

In [1]:
%pip install lightningrod-ai python-dotenv

from IPython.display import clear_output
clear_output()

## Set up the client

Use your API key from [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) (`LIGHTNINGROD_API_KEY` in the environment or Colab secrets).

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

## Prepare the dataset

Use a dataset ID from a pipeline that produced **labeled** samples (e.g. Q&A or content-learning pipelines). Set `LIGHTNINGROD_DATASET_ID` or paste an ID below.

In [3]:
dataset_id = config.get_config_value("LIGHTNINGROD_DATASET_ID")

dataset = lr.datasets.get(dataset_id)
_ = dataset.download()

In [4]:
from lightningrod import prepare_for_training, FilterParams, SplitParams

train_dataset, test_dataset = prepare_for_training(
    dataset,
    filter=FilterParams(),
    split=SplitParams(test_size=0.2, strategy="random"),
)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> prepare_for_training                                                                                        │
│                                                                                                                 │
│    Starting with 375 samples                                                                                    │
│                                                                                                                 │
│    Filter:  Dropped 2 invalid → 373 remain                                                                      │
│    Dedup:   373 remain (0 duplicates)                                                                           │
│    Split:   Splits: 298 train | 75 test (0 dropped, no prediction_date)                                         │
│                                                                                                                 │
│  ⚠ Unhealthy dataset                                                                                            │
│                                                                                                                 │
│  Only 298 train samples remain after preparation. This is below the recommended minimum of +1000 for effective  │
│  training.                                                                                                      │
│                                                                                                                 │
│    Tips:                                                                                                        │
│      • Increase max_seeds in lr.transforms.run() to generate more samples.                                  │
│      • Increase questions_per_seed in your question generator (ForwardLookingQuestionGenerator or               │
│  QuestionGenerator) to produce more questions from each seed article.Add more search queries to your seed       │
│  generator to diversify seed sources.                                                                           │
│      • Widen the seed generator date range (start_date to end_date) to capture more events.                     │
│                                                                                                                 │
│  Only 75 test samples remain after preparation. This is below the recommended minimum of +200 for reliable      │
│  evaluation.                                                                                                    │
│                                                                                                                 │
│    Tips:                                                                                                        │
│      • Generate more samples overall — test samples come from the most recent portion of your date range.       │
│      • Ensure your seed generator date range extends close to the present so recent events appear in the test   │
│  set.                                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Estimate cost and run SFT

`SFTTrainingConfig` supports `epochs`, `resume_from`, and the usual LoRA hyperparameters. It does **not** use `num_rollouts` or `max_response_length` (those are GRPO-only).

In [5]:
from lightningrod import SFTTrainingConfig

sft_config = SFTTrainingConfig(
    base_model_id="openai/gpt-oss-120b",
    training_steps=50,
    epochs=1,
    learning_rate=2e-4,
)

> Note: cost estimation (`lr.training.estimate_cost`) is not yet supported for SFT.

In [6]:
job = lr.training.run(sft_config, dataset=train_dataset, name="SFT fine-tune")
print(f"Status: {job.status}, model_id: {job.model_id}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Training COMPLETED                                                                                          │
│                                                                                                                 │
│    Job ID: 61034bfd-96b3-44cd-855f-edd28b622913                                                                 │
│                                                                                                                 │
│    Model: checkpoint:61034bfd-96b3-44cd-855f-edd28b622913                                                       │
│                                                                                                                 │
│    loss: latest 0.0761  avg 1.0559  (10 steps)  (lower is better)                                               │
│        █▅▁▁▁▁▁▁▁▁                                                                                               │
│    learning_rate: latest 0.0000  avg 0.0001  (10 steps)                                                         │
│        ██▇▆▅▅▄▃▂▁                                                                                               │
│    mean_train_tokens: latest 1067.5000  avg 1084.4031  (10 steps)                                               │
│        ▁▅██▂▅▅▂▆▃                                                                                               │
│                                                                                                                 │
│    Cost:  $0.23                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Status: COMPLETED, model_id: checkpoint:61034bfd-96b3-44cd-855f-edd28b622913


### Evaluation (SFT)

`lr.evals.run_from_training_job()` is not supported for **SFT** yet: the SDK raises `NotImplementedError` because evaluation metrics for SFT are still being implemented. For **GRPO**, `run_from_training_job()` automatically benchmarks the base model, your fine-tuned checkpoint, and any optional `extra_models`; use `lr.evals.run(dataset, models)` for a custom model list.

To run an eval job with a fully custom model list while using SFT, use `lr.evals.create(...)` with explicit `EvalModel` entries — see [Evaluation](../../docs/fine-tuning/evaluation.md).

## Next steps

- Full field reference: [Training](../../docs/fine-tuning/training.md) (in-repo) or your doc site.
- Richer SFT dataset patterns: `.claude/skills/content-learning-examples/SKILL.md` in this repository.